# 01 — Download, verify & extract OSCD

Fetches the three OSCD zips from the Hugging Face mirror into Drive (first run only, ~488 MB total), extracts them to scratch, and runs the layout-discovery parser to summarize what it found.

**Storage contract** — code lives on GitHub, data & checkpoints live on Drive (`MyDrive/s2cd`), `/content` is disposable scratch. Every notebook is self-contained: run all cells top-to-bottom in a fresh GPU session.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import os

REPO_URL = "https://github.com/Aadith-Geeth-Mohan/sentinel2-change-detection.git"
REPO = "/content/s2-change-detection"
if os.path.exists(REPO):
    !git -C {REPO} pull --ff-only
else:
    !git clone {REPO_URL} {REPO}
%cd {REPO}
!pip install -q .
import sys
sys.path.insert(0, os.path.join(REPO, "src"))
# drop cached s2cd modules so a just-pulled fix takes effect in THIS kernel
for k in [k for k in list(sys.modules) if k == "s2cd" or k.startswith("s2cd.")]:
    del sys.modules[k]


In [ ]:
import torch
from s2cd.data import DriveStore
from s2cd.utils import get_device, describe_device, load_config

device = get_device()
print(describe_device(device))
store = DriveStore()  # Drive root: MyDrive/s2cd · scratch: /content


In [ ]:
# One-time download: fetch the 3 OSCD zips from the Hugging Face mirror
# straight into Drive (idempotent — files already complete are skipped).
# Exact byte sizes catch partial downloads before extraction ever runs.
import os

base = "https://hf.co/datasets/hkristen/oscd/resolve/4958d786c1389ede1511d91a6ecf1a75c4074933"
names = ["Onera Satellite Change Detection dataset - Images.zip",
         "Onera Satellite Change Detection dataset - Train Labels.zip",
         "Onera Satellite Change Detection dataset - Test Labels.zip"]
expected = {"Images": 512716711, "Train": 137873, "Test": 83614}  # bytes
dest = "/content/drive/MyDrive/s2cd/raw/oscd"
os.makedirs(dest, exist_ok=True)
for n in names:
    key = "Train" if "Train" in n else "Test" if "Test" in n else "Images"
    out = os.path.join(dest, n)
    if os.path.exists(out) and os.path.getsize(out) == expected[key]:
        print(f"already on Drive: {n}"); continue
    !wget -q -O "{out}" "{base}/{n.replace(' ', '%20')}"
    ok = os.path.getsize(out) == expected[key]
    print(f"{'OK' if ok else 'SIZE MISMATCH'}: {n} ({os.path.getsize(out)} bytes)")
!ls -la "{dest}"


In [ ]:
zips = store.oscd_zips()  # raises with instructions if missing
for z in zips:
    print(f"{z.name}: {z.stat().st_size / 1e9:.2f} GB")


In [ ]:
oscd_root = store.extract_oscd()  # idempotent per session
print(oscd_root)


In [ ]:
from s2cd.data.oscd import list_cities, read_city

cities = list_cities(oscd_root)
print("train cities:", len(cities["train"]), sorted(cities["train"]))
print("test cities: ", len(cities["test"]), sorted(cities["test"]))

bands = load_config("configs/data.yaml")["bands"]
sample = sorted(cities["train"])[0]
city = read_city(oscd_root, sample, bands)
print(f"\nsample city {sample!r}: t0 {city['t0'].shape}, "
      f"dates {city['dates'][0]} -> {city['dates'][1]}")
print("change pixels:", int(city["mask"].sum()), "/", city["mask"].size)
